LLM Gateway

In [1]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

from litellm import completion

In [2]:
import litellm
litellm.suppress_debug_info = True

In [7]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [5]:
response_groq = completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role":"user","content":"Explain RAG in one sentence"}]
)
print(f"Groq:  {response_groq.choices[0].message.content}")

Groq:  RAG (Retrieval-Augmented Generation) is a type of artificial intelligence model that generates text by combining the retrieval of relevant information from a knowledge base with the generation of new text based on that information.


In [8]:
from litellm import completion

prompt = "Explain RAG in one sentence."

providers = [
    ("Groq","groq/llama-3.3-70b-versatile"),
    ("Gemini","gemini/gemini-1.5-flash"),
    ("Anthropic","claude-3-5-haiku-20241022")
]

for label, model in providers:
    try:
        r = completion(model=model,messages=[{"role":"user","content":prompt}])
        print(f"{label}: {r.choices[0].message.content[:80]}")
    except Exception as e:
        print(f"{label}: {type(e).__name__}")

Groq: RAG (Retrieve, Augment, Generate) is a framework for text generation that levera
Gemini: NotFoundError
Anthropic: BadRequestError


Automatic FallBacks

In [9]:
response = completion(
    model="gemini/gemini-1.5-flash",
    messages=[{"role":"user","content":"What is an LLM Gateway?"}],
    fallbacks=[
        "gpt-4o-mini",
        "groq/llama-3.3-70b-versatile"
    ]
)
print("Response:", response.choices[0].message.content[:200])
print("Model who answered: ", response.model)

17:40:17 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model gemini/gemini-1.5-flash: litellm.NotFoundError: GeminiException - {
  "error": {
    "code": 404,
    "message": "models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.",
    "status": "NOT_FOUND"
  }
}
Traceback (most recent call last):
  File "C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\litellm\llms\vertex_ai\gemini\vertex_and_google_ai_studio_gemini.py", line 2870, in async_completion
    response = await client.post(
               ^^^^^^^^^^^^^^^^^^
  File "C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\litellm\litellm_core_utils\logging_utils.py", line 289, in async_wrapper
    result = await func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\litellm

Response: An LLM (Large Language Model) Gateway is an interface or a software layer that enables access to and interaction with large language models (LLMs) from various applications, systems, or services. The 
Model who answered:  llama-3.3-70b-versatile


Cost Tracking

In [14]:
from litellm import completion, completion_cost

response = completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role":"user","content":"Write a haiku about AI"}]
)

cost = completion_cost(completion_response=response,model="groq/llama-3.3-70b-versatile")
print("Response: ",response.choices[0].message.content)
print("Input tokens: ",response.usage.prompt_tokens)
print("Output tokens: ",response.usage.completion_tokens)
print(f"Cost: ${cost:.8f}")

Response:  Metal mind awakes
Thinking, learning, growing fast
Humanity's aid
Input tokens:  41
Output tokens:  17
Cost: $0.00003762


Caching

In [15]:
import litellm

litellm.callback = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []

litellm.cache = None

In [16]:
import time
from litellm.caching import Cache

litellm.cache = Cache(type="local")

prompt = "What does LLM stand for? Answer in one line"

start = time.time()
r1 = completion(
    model = "groq/llama-3.3-70b-versatile",
    messages=[{"role":"user","content":prompt}],
    caching = True
)
t1 = time.time() - start
print(f"First call: {t1:.2f}s - {r1.choices[0].message.content}")

start = time.time()
r2 = completion(
    model = "groq/llama-3.3-70b-versatile",
    messages=[{"role":"user","content":prompt}],
    caching = True
)
t2 = time.time() - start
print(f"Second call: {t2:.4f}s - {r2.choices[0].message.content}")


First call: 7.60s - LLM stands for Large Language Model, a type of artificial intelligence (AI) designed to process and understand human language.
Second call: 0.0038s - LLM stands for Large Language Model, a type of artificial intelligence (AI) designed to process and understand human language.


Smart Routing

use litellm's router to define routing rules

In [17]:
from litellm import Router

model_list = [
    {
        "model_name":"fast-cheap",
        "litellm_params":{
            "model":"groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name":"smart-coding",
        "litellm_params":{
            "model":"gpt-4o",
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    },
    {
        "model_name":"balanced",
        "litellm_params":{
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    }
]
router = Router(model_list=model_list)

fast_response = router.completion(
    model="fast-cheap",
    messages=[{"role":"user","content":"Summarize: AI is changing software"}]
)

print(f"Fast Response: {fast_response.choices[0].message.content[:150]}")

Fast Response: The advent of Artificial Intelligence (AI) is revolutionizing the software industry in several ways. Here's a summary of the key changes:

1. **Automa


Load Balancing across multiple API keys

hit rate limits on one Provider of API Key, add more keys to same alias the router load-balances automatically

In [19]:
model_list = [
    {
        "model_name":"gpt-pool",
        "litellm_params":{
            "model":"gpt-4o",
            "api_key": os.getenv("OPENAI_API_KEY")
        },
        "model_info": {"id":"openai-gpt4o"}
    },
    {
        "model_name":"gpt-pool",
        "litellm_params":{
            "model":"groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY")
        },
        "model_info": {"id":"groq-llama-70b"}
    }
]

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"
)

print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency':<12}{'Response':<40}")
print("-"*84)
for i in range(6):
    r = router.completion(
        model = "gpt-pool",
        messages=[{"role":"user","content":f"Say Hello, request {i+1}"}]
    )
    deployment_id = r._hidden_params.get("model_id","unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content[:35]
    print(f"#{i+1:<9}{deployment_id:<22}{latency:>6.0f} ms {answer}")

Request   Deployment Picked     Latency     Response                                
------------------------------------------------------------------------------------
#1        groq-llama-70b          1192 ms Hello. This is request 1. How can I
#2        groq-llama-70b           238 ms Hello. This is request 2. How can I
#3        groq-llama-70b           288 ms Hello. You've made request number 3
#4        groq-llama-70b           491 ms Hello. You've made request number 4
#5        groq-llama-70b           388 ms Hello. You've requested 5 of someth
#6        groq-llama-70b           400 ms Hello. You've made request number 6


there are multiple strategies
- simple-shuffle
- least-busy (uses the least busy api key)
- latency-based-routing (measures response time and picks the fastest)